# bu v2.pth modeli vardı onunla map hesabı yaptım ilk. sonuçları aşağıda

In [ ]:
import os
import sys
import torch
import torchvision
import json
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from pprint import pprint

# 1. KURULUM
!{sys.executable} -m pip install torchmetrics[detection] pycocotools

# 2. AYARLAR
# r"..." kullanarak Windows yollarını güvenli hale getir
BASE_PATH = r"C:\Users\LENOVO\Downloads\GP Object detection.v1i.coco"
MODEL_PATH = "gp_food_model_v2.pth"
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# 3. DATASET SINIFI (Roboflow/COCO Formatı)
class FoodMAPDataset(Dataset):
    def __init__(self, root, ann_file, transform=None):
        self.root = root
        self.transform = transform
        with open(ann_file, 'r') as f:
            self.coco_data = json.load(f)
        self.images = self.coco_data['images']
        self.annotations = self.coco_data['annotations']

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.root, img_info['file_name'])
        img = Image.open(img_path).convert("RGB")
        
        # Resme ait açıklamaları bul
        ann_list = [a for a in self.annotations if a['image_id'] == img_info['id']]
        boxes, labels = [], []
        for ann in ann_list:
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'] + 1) # Background (0) dengelemek için +1
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32).view(-1, 4)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            # Boş resimler için güvenli tensorlar
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([img_info['id']])}

        if self.transform:
            img = self.transform(img)
            # Kutuları 224x224 boyutuna normalize et
            orig_w, orig_h = img_info['width'], img_info['height']
            target['boxes'][:, [0, 2]] *= (224 / orig_w)
            target['boxes'][:, [1, 3]] *= (224 / orig_h)

        return img, target

    def __len__(self):
        return len(self.images)

def collate_fn(batch):
    return tuple(zip(*batch))

# 4. YÜKLEME VE ÇALIŞTIRMA
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validasyon Loader
valid_root = os.path.join(BASE_PATH, "valid")
valid_ann = os.path.join(valid_root, "_annotations.coco.json")

# Dosya var mı kontrol et
if not os.path.exists(valid_ann):
    print(f"HATA: Dosya bulunamadı -> {valid_ann}")
else:
    valid_ds = FoodMAPDataset(valid_root, valid_ann, transform=test_transform)
    validation_loader = DataLoader(valid_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

    # Model Yükleme
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 21) # 20 Yemek + 1 Arkaplan
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE).eval()

    # mAP Hesaplama
    metric = MeanAveragePrecision(iou_type="bbox")
    print("Değerlendirme başladı...")
    with torch.no_grad():
        for images, targets in validation_loader:
            images = list(img.to(DEVICE) for img in images)
            outputs = model(images)
            res = [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outputs]
            target_res = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
            metric.update(res, target_res)

    results = metric.compute()
    print("\n--- FINAL mAP SONUÇLARI ---")
    pprint(results)

Değerlendirme başladı...

--- FINAL mAP SONUÇLARI ---
{'classes': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20], dtype=torch.int32),
 'map': tensor(0.3220),
 'map_50': tensor(0.6462),
 'map_75': tensor(0.2602),
 'map_large': tensor(0.5474),
 'map_medium': tensor(0.3390),
 'map_per_class': tensor(-1.),
 'map_small': tensor(0.2641),
 'mar_1': tensor(0.4167),
 'mar_10': tensor(0.4287),
 'mar_100': tensor(0.4287),
 'mar_100_per_class': tensor(-1.),
 'mar_large': tensor(0.6081),
 'mar_medium': tensor(0.4420),
 'mar_small': tensor(0.3104)}


# burda finetuned.pth ile yaptığım map sonucu

In [1]:
import os
import sys
import torch
import torchvision
import json
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from pprint import pprint

# 1. KURULUM
!{sys.executable} -m pip install torchmetrics[detection] pycocotools

# 2. AYARLAR
# r"..." kullanarak Windows yollarını güvenli hale getir
BASE_PATH = r"C:\Users\LENOVO\Downloads\GP Object detection.v1i.coco"
MODEL_PATH = "gp_food_model_finetuned.pth"
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# 3. DATASET SINIFI (Roboflow/COCO Formatı)
class FoodMAPDataset(Dataset):
    def __init__(self, root, ann_file, transform=None):
        self.root = root
        self.transform = transform
        with open(ann_file, 'r') as f:
            self.coco_data = json.load(f)
        self.images = self.coco_data['images']
        self.annotations = self.coco_data['annotations']

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.root, img_info['file_name'])
        img = Image.open(img_path).convert("RGB")
        
        # Resme ait açıklamaları bul
        ann_list = [a for a in self.annotations if a['image_id'] == img_info['id']]
        boxes, labels = [], []
        for ann in ann_list:
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'] + 1) # Background (0) dengelemek için +1
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32).view(-1, 4)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            # Boş resimler için güvenli tensorlar
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([img_info['id']])}

        if self.transform:
            img = self.transform(img)
            # Kutuları 224x224 boyutuna normalize et
            orig_w, orig_h = img_info['width'], img_info['height']
            target['boxes'][:, [0, 2]] *= (224 / orig_w)
            target['boxes'][:, [1, 3]] *= (224 / orig_h)

        return img, target

    def __len__(self):
        return len(self.images)

def collate_fn(batch):
    return tuple(zip(*batch))

# 4. YÜKLEME VE ÇALIŞTIRMA
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validasyon Loader
valid_root = os.path.join(BASE_PATH, "valid")
valid_ann = os.path.join(valid_root, "_annotations.coco.json")

# Dosya var mı kontrol et
if not os.path.exists(valid_ann):
    print(f"HATA: Dosya bulunamadı -> {valid_ann}")
else:
    valid_ds = FoodMAPDataset(valid_root, valid_ann, transform=test_transform)
    validation_loader = DataLoader(valid_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

    # Model Yükleme
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 21) # 20 Yemek + 1 Arkaplan
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE).eval()

    # mAP Hesaplama
    metric = MeanAveragePrecision(iou_type="bbox")
    print("Değerlendirme başladı...")
    with torch.no_grad():
        for images, targets in validation_loader:
            images = list(img.to(DEVICE) for img in images)
            outputs = model(images)
            res = [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outputs]
            target_res = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
            metric.update(res, target_res)

    results = metric.compute()
    print("\n--- FINAL mAP SONUÇLARI ---")
    pprint(results)

Değerlendirme başladı...

--- FINAL mAP SONUÇLARI ---
{'classes': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20], dtype=torch.int32),
 'map': tensor(0.3388),
 'map_50': tensor(0.6672),
 'map_75': tensor(0.3312),
 'map_large': tensor(0.6144),
 'map_medium': tensor(0.3601),
 'map_per_class': tensor(-1.),
 'map_small': tensor(0.2312),
 'mar_1': tensor(0.4234),
 'mar_10': tensor(0.4391),
 'mar_100': tensor(0.4395),
 'mar_100_per_class': tensor(-1.),
 'mar_large': tensor(0.6400),
 'mar_medium': tensor(0.4631),
 'mar_small': tensor(0.2729)}
